# Heat + Burgers: baseline exploratorio instrumentado

Esta libreta reproduce el manifiesto y separa preparación, preflight, muestreo,
entrenamiento, medición, visualización y cierre. Conserva el protocolo científico
previo y cuantifica señales por problema y semilla; no afirma SOTA.

| Etapa | Elección |
| --- | --- |
| Entrenamiento | PINA + PyTorch, con RBF-PIKAN instrumentada |
| Seguimiento | W&B y artefactos locales reproducibles |
| Exploración de señales y curvas | Plotly |
| Campos, geometrías y mallas | PyVista, con exportación sin ventana |
| Aplicación de análisis posterior | Panel + Plotly/PyVista |

Heat y Burgers tienen campos espacio-tiempo 1D, así que sus mapas son Plotly.
PyVista se reserva para geometrías y mallas explícitas, como DFG/Navier–Stokes;
esta campaña no genera una malla 3D artificial. W&B trabaja offline durante la
corrida y conserva datos locales para sincronización autenticada posterior.


## 1. Reproducibilidad y carga del paquete

La libreta incluye fuentes versionadas y sus hashes; no clona Git ni embebe secretos.


In [ ]:
import base64, io, os, pathlib, sys, zipfile, zlib

BUNDLE_ARCHIVE_B64 = (
    '...'
)
BUNDLE_ID = '730cc2b20e55d5ed'
default_work = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else pathlib.Path.cwd()
WORK_ROOT = pathlib.Path(os.environ.get('KAGGLE_WORKING_DIR', str(default_work))) / ('pikan-' + BUNDLE_ID)
BUNDLE_DIR = WORK_ROOT / 'bundle'
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
archive_bytes = zlib.decompress(base64.b64decode(BUNDLE_ARCHIVE_B64))
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for item in archive.infolist():
        relative = pathlib.PurePosixPath(item.filename)
        if relative.is_absolute() or '..' in relative.parts or '\\' in item.filename:
            raise ValueError('Unsafe bundle path: ' + item.filename)
        target = (BUNDLE_DIR / pathlib.Path(*relative.parts)).resolve()
        if not target.is_relative_to(BUNDLE_DIR.resolve()):
            raise ValueError('Bundle path escaped destination: ' + item.filename)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(item))
sys.path.insert(0, str(BUNDLE_DIR))
from runtime import verify_bundle
SPEC = verify_bundle(BUNDLE_DIR)
assert SPEC['bundle_id'].startswith(BUNDLE_ID)
RUNS_DIR = WORK_ROOT / 'runs'
ANALYSIS_DIR = WORK_ROOT / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print('Bundle verified:', SPEC['bundle_id'], 'source files:', len(SPEC['files']))


## 2. Dependencias y versiones

PyTorch/CUDA se comprueban contra Kaggle. Las dependencias restantes se instalan
solo si falta un pin o su versión difiere. Panel y PyVista no se instalan en el
entrenamiento porque estos dominios 1D no los necesitan.


In [ ]:
import importlib.metadata, subprocess, sys

DEPENDENCIES = ['pina-mathlab==0.3.2.post2609', 'wandb==0.30.0', 'plotly==7.1.0', 'scipy==1.15.3', 'pandas==2.2.3']
missing_or_different = []
for requirement in DEPENDENCIES:
    name, expected = requirement.split('==', 1)
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        missing_or_different.append(requirement)
if missing_or_different:
    command = [sys.executable, '-m', 'pip', 'install',
               '--disable-pip-version-check', *missing_or_different]
    print('Installing pinned dependencies:', ', '.join(missing_or_different), flush=True)
    subprocess.run(command, check=True, timeout=900)
else:
    print('Pinned experiment dependencies already match the manifest.')


## 3. Protocolo y decisiones de parámetros

El manifiesto es la fuente de verdad para condiciones, arquitectura, muestreo,
semillas, pasos, tiempo, diagnóstico y versiones. Las tablas reportan valores
calculados desde artefactos; las decisiones completas se conservan junto a resultados.


## 4. Preflight: dos T4 y asignación de semillas

Se exige que Kaggle exponga dos T4 antes de entrenar. Una semilla ocupa una GPU y
entrena Heat seguido de Burgers; la tercera comienza en la primera GPU libre. Cada
worker ve una sola GPU lógica mediante CUDA_VISIBLE_DEVICES.


In [ ]:
import platform, subprocess, torch
import pandas as pd
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable; the approved T4x2 campaign did not start.')
visible_gpus = torch.cuda.device_count()
gpu_names = [torch.cuda.get_device_name(index) for index in range(visible_gpus)]
if visible_gpus < 2:
    raise RuntimeError(f'T4x2 was requested, but only {visible_gpus} CUDA devices are visible: {gpu_names}')
if any('T4' not in name.upper() for name in gpu_names[:2]):
    raise RuntimeError(f'The requested accelerator is 2xT4; visible devices are {gpu_names[:2]}')
GPU_SLOTS = ['0', '1']
hardware = pd.DataFrame([
    {'logical_gpu': i, 'name': gpu_names[i],
     'memory_total_mib': round(torch.cuda.get_device_properties(i).total_memory / 2**20),
     'parallel_seed_slot': GPU_SLOTS[i]} for i in range(2)
])
display(hardware)
display(pd.DataFrame([{'python': platform.python_version(), 'pytorch': torch.__version__,
                       'cuda_runtime': torch.version.cuda, 'visible_gpu_count': visible_gpus}]))
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'],
               check=True, text=True)


## 5. Generación y cobertura de puntos

La tabla presenta conteos generados, discrepancia centrada de Sobol y distancias
al vecino más cercano en coordenadas normalizadas. La figura muestra los puntos
generados realmente para la primera semilla.


In [ ]:
import json, numpy as np, pandas as pd, plotly.express as px
from scipy.spatial import cKDTree
from scipy.stats import qmc
from IPython.display import display
from src.research_pipeline.sciml.sampling import pools

manifest = json.loads((BUNDLE_DIR / 'manifest.json').read_text(encoding='utf-8'))
cfg = manifest['training']
sampling_rows = []
sampling_points = {}
for seed in cfg['seeds']:
    for problem in manifest['problem']['sequence']:
        effective = {**cfg, **cfg.get('problem_overrides', {}).get(problem, {})}
        data = pools(problem, seed, effective['interior_points'], effective['boundary_points'])
        points = data['interior'].copy()
        points[:, 0] = (points[:, 0] - (0.0 if problem == 'heat' else -1.0)) / (1.0 if problem == 'heat' else 2.0)
        neighbors = cKDTree(points).query(points, k=2)[0][:, 1]
        sampling_rows.append({
            'problem': problem, 'seed': seed, 'interior_n': len(data['interior']),
            'diagnostic_n': len(data['diagnostic']), 'boundary_n': len(data['left']) + len(data['right']),
            'initial_n': len(data['initial']), 'sobol_centered_discrepancy': qmc.discrepancy(points, method='CD'),
            'nn_min': np.min(neighbors), 'nn_median': np.median(neighbors),
            'nn_p95': np.quantile(neighbors, .95), 'nn_max': np.max(neighbors),
        })
        if seed == cfg['seeds'][0]:
            sampling_points[problem] = data['interior']
sampling_table = pd.DataFrame(sampling_rows)
display(sampling_table.style.format(precision=6))
sampling_table.to_csv(ANALYSIS_DIR / 'sampling-signals.csv', index=False)
for problem, points in sampling_points.items():
    figure = px.scatter(x=points[::4, 0], y=points[::4, 1], opacity=.55,
                        labels={'x': 'space coordinate x', 'y': 'time coordinate t'},
                        title=f'{problem.title()} interior Sobol pool, seed {cfg["seeds"][0]}')
    figure.write_html(ANALYSIS_DIR / f'{problem}-sampling.html', include_plotlyjs='cdn')
    display(figure)


## 6. Entrenamiento concurrente PINA + PyTorch + RBF-PIKAN

Este es el único bloque prolongado. Usa el runtime reproducible del repositorio,
con workers por semilla y asignación explícita de GPU. La curva se registra cada
100 pasos, los diagnósticos físicos/RBF cada 250 y un checkpoint final por problema.
El límite de pasos o tiempo no implica convergencia.


In [ ]:
from runtime import run_bundle

MAX_PARALLEL_SEEDS = SPEC['max_parallel_seeds']
if len(GPU_SLOTS) != MAX_PARALLEL_SEEDS:
    raise RuntimeError('GPU slot assignment does not match approved parallelism.')
seed_order = json.loads((BUNDLE_DIR / 'manifest.json').read_text(encoding='utf-8'))['training']['seeds']
print('Seed schedule:', {seed: GPU_SLOTS[index] for index, seed in enumerate(seed_order[:MAX_PARALLEL_SEEDS])},
      '; later seeds use the first free T4.', flush=True)
campaign_state = run_bundle(BUNDLE_DIR, RUNS_DIR, max_parallel=MAX_PARALLEL_SEEDS,
                            gpu_slots=GPU_SLOTS)
display(pd.DataFrame([{'seed': seed, 'status': status}
                      for seed, status in sorted(campaign_state['seeds'].items())]))
if any(status != 'completed' for status in campaign_state['seeds'].values()):
    raise RuntimeError('Campaign ended with incomplete seeds: ' + repr(campaign_state['seeds']))


## 7. Tabla de señales cuantificadas

Cada fila es un problema y una semilla. Las columnas informan mediciones de puntos,
pérdidas, gradientes, autodiferenciación, ocupación RBF, saturación, sensibilidad,
validación y recursos. "not measured" significa ausente, nunca cero.


In [ ]:
from src.research_pipeline.remote.reporting import (
    aggregate_quantitative_table, long_quantitative_table, quantitative_table)
from IPython.display import display

signals = quantitative_table(RUNS_DIR)
long_signals = long_quantitative_table(signals)
for family in sorted(long_signals['family'].unique()):
    print(f'Measured {family} signals; each value is from a saved run artifact.')
    display(long_signals.loc[long_signals['family'] == family]
            .sort_values(['problem', 'seed', 'signal']).style.format({'value': '{:.6g}'}))
aggregated_signals = aggregate_quantitative_table(signals)
display(aggregated_signals.style.format(precision=6, na_rep='not measured'))
signals.to_csv(ANALYSIS_DIR / 'quantitative-signals.csv', index=False)
long_signals.to_csv(ANALYSIS_DIR / 'quantitative-signals-with-units.csv', index=False)
aggregated_signals.to_csv(ANALYSIS_DIR / 'signal-distributions.csv')


## 8. Curvas de aprendizaje y señales internas

Las curvas salen de history.jsonl y de diagnósticos periódicos guardados. Plotly
muestra pérdidas, norma de gradiente y señales de residual durante el entrenamiento.


In [ ]:
from src.research_pipeline.remote.reporting import training_figures
from IPython.display import display

for name, figure in training_figures(RUNS_DIR).items():
    figure.write_html(ANALYSIS_DIR / f'{name}.html', include_plotlyjs='cdn')
    display(figure)


## 9. Campos de validación independiente

Los mapas de campo y error usan los arrays de validación guardados. La malla es
independiente de los puntos Sobol empleados para entrenar.


In [ ]:
from src.research_pipeline.remote.reporting import field_figures
from IPython.display import display

for name, figure in field_figures(RUNS_DIR).items():
    figure.write_html(ANALYSIS_DIR / f'{name}.html', include_plotlyjs='cdn')
    display(figure)


## 10. Cierre, hashes y paquete de resultados

La celda final conserva decisiones, hashes, CSV y figuras en un archivo de resultados.
El verificador mantiene la autoridad sobre las métricas; ningún resultado queda marcado
como aceptado.


In [ ]:
import hashlib, json, zipfile
from src.research_pipeline.remote.reporting import write_results_notebook

manifest = json.loads((BUNDLE_DIR / 'manifest.json').read_text(encoding='utf-8'))
(ANALYSIS_DIR / 'parameter-decisions.md').write_text(
    manifest['parameter_decisions_markdown'], encoding='utf-8')
results_notebook = write_results_notebook(RUNS_DIR, ANALYSIS_DIR, SPEC['bundle_id'])
files = sorted(path for path in WORK_ROOT.rglob('*') if path.is_file()
               and not any(part.startswith(('wandb', '.')) for part in path.relative_to(WORK_ROOT).parts))
hashes = {path.relative_to(WORK_ROOT).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
          for path in files}
hash_path = ANALYSIS_DIR / 'artifact-hashes.json'
hash_path.write_text(json.dumps(hashes, indent=2), encoding='utf-8')
archive_path = WORK_ROOT / 'heat-burgers-t4x2-results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files + [hash_path]:
        archive.write(path, path.relative_to(WORK_ROOT).as_posix())
print('Run closed. Results archive:', archive_path, '; files hashed:', len(hashes))
print('Exploratory measurements only; no SOTA or scientific claim is established.')
